In [1]:
!pip install pytdc transformers rdkit fair-esm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.2/154.2 kB 7.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.5 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of pytdc to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.3/151.3 kB 9.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 60.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.2/151.2 kB 12.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.2/151.2 kB 16.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... d

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
import random
import pickle
import copy
import math
import time
import esm
from rdkit import Chem
from rdkit.Chem import QED
from rdkit.Chem import AllChem, DataStructs
from rdkit.Contrib.SA_Score import sascorer
from scipy.stats import pearsonr, spearmanr
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModel
from sklearn.metrics import roc_auc_score, average_precision_score
from tdc.multi_pred import DTI
from tqdm.auto import tqdm
from pathlib import Path

In [2]:
SEED = 42
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [3]:
print("Dataset: KIBA")
DATASET_NAME = "KIBA"
dti_data = DTI(name=DATASET_NAME)
kiba_data = dti_data.get_data()
kiba_data

Downloading...


Dataset: KIBA


100%|██████████| 96.6M/96.6M [00:02<00:00, 33.2MiB/s]
Loading...
Done!


,Drug_ID,Drug,Target_ID,Target,Y
0,CHEMBL1087421,COc1cc2c(cc1Cl)C(c1ccc(Cl)c(Cl)c1)=NCC2,O00141,MTVKTEAAKGTLTYSRMRGMVAILIAFMKQRRMGLNDFIQKIANNS...,11.10000
1,CHEMBL1087421,COc1cc2c(cc1Cl)C(c1ccc(Cl)c(Cl)c1)=NCC2,O14920,MSWSPSLTTQTCGAWEMKERLGTGGFGNVIRWHNQETGEQIAIKQC...,11.10000
2,CHEMBL1087421,COc1cc2c(cc1Cl)C(c1ccc(Cl)c(Cl)c1)=NCC2,O15111,MERPPGLRPGAGGPWEMRERLGTGGFGNVCLYQHRELDLKIAIKSC...,11.10000
3,CHEMBL1087421,COc1cc2c(cc1Cl)C(c1ccc(Cl)c(Cl)c1)=NCC2,P00533,MRPSGTAGAALLALLAALCPASRALEEKKVCQGTSNKLTQLGTFED...,11.10000
4,CHEMBL1087421,COc1cc2c(cc1Cl)C(c1ccc(Cl)c(Cl)c1)=NCC2,P04626,MELAALCRWGLLLALLPPGAASTQVCTGTDMKLRLPASPETHLDML...,11.10000
...,...,...,...,...,...
117652,CHEMBL230654,CCCc1nc[nH]c1CNc1cc(Cl)c2ncc(C#N)c(Nc3ccc(F)c(...,Q13554,MATTVTCTRFTDEYQLYEDIGKGAFSVVRRCVKLCTGHEYAAKIIN...,10.49794
117653,CHEMBL230654,CCCc1nc[nH]c1CNc1cc(Cl)c2ncc(C#N)c(Nc3ccc(F)c(...,Q13555,MATTATCTRFTDDYQLFEELGKGAFSVVRRCVKKTSTQEYAAKIIN...,10.49794
117654,CHEMBL230654,CCCc1nc[nH]c1CNc1cc(Cl)c2ncc(C#N)c(Nc3ccc(F)c(...,Q13557,MASTTTCTRFTDEYQLFEELGKGAFSVVRRCMKIPTGQEYAAKIIN...,10.49794
117655,CHEMBL230654,CCCc1nc[nH]c1CNc1cc(Cl)c2ncc(C#N)c(Nc3ccc(F)c(...,Q16539,MSQERPTFYRQELNKTIWEVPERYQNLSPVGSGAYGSVCAAFDTKT...,10.49794


In [4]:
FRAC = [0.7, 0.1, 0.2]
splits = {
    "cold_drug": dti_data.get_split(method="cold_split", column_name="Drug", seed=SEED, frac=FRAC)
}

for split_name, split in splits.items():
    print(f"\n[{split_name}]")
    for part in ["train", "valid", "test"]:
        print(part, split[part].shape)


[cold_drug]
train (82539, 5)
valid (12023, 5)
test (23095, 5)


In [5]:
def check_overlap(split, column):
    train_set = set(split["train"][column])
    valid_set = set(split["valid"][column])
    test_set = set(split["test"][column])
    return {
        "train_valid_overlap": len(train_set & valid_set),
        "train_test_overlap": len(train_set & test_set),
        "valid_test_overlap": len(valid_set & test_set),
    }

print("Cold drug overlap check on Drug:")
print(check_overlap(splits["cold_drug"], "Drug"))

Cold drug overlap check on Drug:
{'train_valid_overlap': 0, 'train_test_overlap': 0, 'valid_test_overlap': 0}


In [6]:
CACHE_DIR = Path("cache")
CACHE_DIR.mkdir(exist_ok=True)
DRUG_FP_BITS = 1024
DRUG_FP_RADIUS = 2
ESM_REPR_LAYER = 6
ESM_BATCH_SIZE = 8
PROTEIN_MAX_LEN = 2527

In [7]:
def smiles_to_morgan_count_vec(smiles, radius=2, n_bits=1024):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return np.zeros(n_bits, dtype=np.float32)

    fp = AllChem.GetMorganFingerprint(mol, radius)
    arr = np.zeros((n_bits,), dtype=np.float32)
    for bit, count in fp.GetNonzeroElements().items():
        idx = bit % n_bits
        arr[idx] += count

    return arr.astype(np.float32)

print("Building Morgan COUNT fingerprints...")
drug_features = {}
unique_smiles = sorted(kiba_data["Drug"].unique())

for smiles in tqdm(unique_smiles):
    drug_features[smiles] = smiles_to_morgan_count_vec(
        smiles,
        radius=DRUG_FP_RADIUS,
        n_bits=DRUG_FP_BITS,
    )

DRUG_DIM = next(iter(drug_features.values())).shape[0]
ex_drug_vec = next(iter(drug_features.values()))
print("Drug dim:", DRUG_DIM)
print("First Drug embedding [:200]:", ex_drug_vec[:200])

Building Morgan COUNT fingerprints...


  0%|          | 0/2068 [00:00<?, ?it/s]

Drug dim: 1024
First Drug embedding [:200]: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 2.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0.]


In [8]:
def build_or_load_protein_esm_embeddings(df, cache_path, batch_size=8):
    if Path(cache_path).exists():
        print(f"Loading cached protein ESM embeddings from {cache_path}")
        with open(cache_path, "rb") as f:
            return pickle.load(f)

    print("Loading ESM-2 model...")
    esm_model, alphabet = esm.pretrained.esm2_t6_8M_UR50D()
    batch_converter = alphabet.get_batch_converter()
    esm_model.eval()
    esm_model = esm_model.to(device)
    unique_targets = sorted(df["Target"].unique())
    protein_embeddings = {}
    print(f"Embedding {len(unique_targets)} unique protein sequences...")

    for start in tqdm(range(0, len(unique_targets), batch_size)):
        batch_original = unique_targets[start:start + batch_size]
        batch_data = [
            (f"protein_{start + i}", seq[:PROTEIN_MAX_LEN])
            for i, seq in enumerate(batch_original)
        ]

        batch_labels, batch_strs, batch_tokens = batch_converter(batch_data)
        batch_tokens = batch_tokens.to(device)

        with torch.no_grad():
            results = esm_model(
                batch_tokens,
                repr_layers=[ESM_REPR_LAYER],
                return_contacts=False,
            )

        token_representations = results["representations"][ESM_REPR_LAYER].detach().cpu()

        for i, original_seq in enumerate(batch_original):
            trunc_len = min(len(original_seq), PROTEIN_MAX_LEN)
            residue_repr = token_representations[i, 1:trunc_len + 1]
            mean_repr = residue_repr.mean(dim=0).numpy().astype(np.float32)
            protein_embeddings[original_seq] = mean_repr

    with open(cache_path, "wb") as f:
        pickle.dump(protein_embeddings, f)

    return protein_embeddings

protein_features = build_or_load_protein_esm_embeddings(
    kiba_data,
    CACHE_DIR / f"{DATASET_NAME.lower()}_protein_esm2_t6_mean_len{PROTEIN_MAX_LEN}.pkl",
    batch_size=ESM_BATCH_SIZE,
)

PROTEIN_DIM = next(iter(protein_features.values())).shape[0]
ex_protein_vec = next(iter(protein_features.values()))
print("Protein dim:", PROTEIN_DIM)
print("First Protein embedding:", ex_protein_vec[:])

Loading ESM-2 model...
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/models/esm2_t6_8M_UR50D.pt" to /root/.cache/torch/hub/checkpoints/esm2_t6_8M_UR50D.pt
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/regression/esm2_t6_8M_UR50D-contact-regression.pt" to /root/.cache/torch/hub/checkpoints/esm2_t6_8M_UR50D-contact-regression.pt
Embedding 229 unique protein sequences...


  0%|          | 0/29 [00:00<?, ?it/s]

Protein dim: 320
First Protein embedding: [ 4.41143624e-02 -2.52826512e-01  2.17878982e-01  1.50879145e-01
  1.93017557e-01 -2.61078805e-01  7.93226287e-02 -8.59324187e-02
 -1.81139231e-01 -2.60318547e-01  2.01544762e-01  4.54776622e-02
 -1.22118354e-01 -5.44682294e-02  1.01339109e-01 -2.40610391e-01
  2.68337931e-02  6.14388799e-03  3.52386795e-02  6.56745359e-02
  6.15391843e-02  1.23918932e-02 -2.92557273e-02 -7.62776006e-03
  1.09924160e-01  9.17342082e-02 -2.57420212e-01  3.94399799e-02
  6.24114461e-02  1.18440375e-01 -6.92011118e-02  2.64154017e-01
  1.05893835e-01 -4.34264511e-01 -1.29478440e-01 -2.57710755e-01
  1.19920127e-01  1.73939213e-01 -4.38638069e-02  5.98966926e-02
  2.18583852e-01  1.42972127e-01  2.11244430e-02 -1.18948780e-01
 -1.64557844e-01  1.81634948e-01 -7.72667527e-01  1.09629288e-01
 -1.69279844e-01 -7.99338147e-02 -4.56359647e-02 -3.08931265e-02
 -9.74544659e-02  8.53029341e-02 -3.86599898e-02 -1.05436921e-01
  2.20586315e-01 -1.16647698e-01  5.61533332e-01

In [9]:
def build_xy(dataframe, drug_features, protein_features):
    drug_x = np.stack([drug_features[smi] for smi in dataframe["Drug"].values]).astype(np.float32)
    protein_x = np.stack([protein_features[seq] for seq in dataframe["Target"].values]).astype(np.float32)
    y = dataframe["Y"].values.astype(np.float32)
    return drug_x, protein_x, y

class DTADataset(Dataset):
    def __init__(self, drug_x, protein_x, y):
        self.drug_x = torch.tensor(drug_x, dtype=torch.float32)
        self.protein_x = torch.tensor(protein_x, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.drug_x[idx], self.protein_x[idx], self.y[idx]

In [10]:
def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    mae = np.mean(np.abs(y_true - y_pred))

    try:
        pearson = pearsonr(y_true, y_pred).statistic
    except Exception:
        pearson = np.nan
    try:
        spearman = spearmanr(y_true, y_pred).statistic
    except Exception:
        spearman = np.nan
    return {
        "RMSE": float(rmse),
        "MAE": float(mae),
        "Pearson": float(pearson),
        "Spearman": float(spearman),
    }

In [11]:
class CrossAttention(nn.Module):
    def __init__(self, drug_dim, protein_dim, hidden_dim=512, dropout=0.2):
        super().__init__()
        self.esm_dim = protein_dim
        self.proj_dim = 256
        self.drug_proj = nn.Sequential(nn.Linear(drug_dim, self.proj_dim), nn.ReLU())
        self.protein_proj = nn.Sequential(nn.Linear(self.esm_dim, self.proj_dim), nn.ReLU())
        self.cross_attn = nn.MultiheadAttention(embed_dim=self.proj_dim, num_heads=4, batch_first=True)
        self.layer_norm_d = nn.LayerNorm(self.proj_dim)
        self.layer_norm_p = nn.LayerNorm(self.proj_dim)
        self.predictor = nn.Sequential(
            nn.Linear(self.proj_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1),
        )

    def forward(self, drug_x, protein_x):
        z_d = self.drug_proj(drug_x)
        z_p = self.protein_proj(protein_x)
        z_d_seq = z_d.unsqueeze(1)
        z_p_seq = z_p.unsqueeze(1)
        attn_d_out, _ = self.cross_attn(query=z_d_seq, key=z_p_seq, value=z_p_seq)
        attn_p_out, _ = self.cross_attn(query=z_p_seq, key=z_d_seq, value=z_d_seq)
        z_d_attended = self.layer_norm_d(z_d_seq + attn_d_out).squeeze(1)
        z_p_attended = self.layer_norm_p(z_p_seq + attn_p_out).squeeze(1)
        fusion = torch.cat([z_d_attended, z_p_attended], dim=1)
        return self.predictor(fusion).squeeze(-1)

In [12]:
def predict(model, loader, device):
    model.eval()
    preds = []
    ys = []

    with torch.no_grad():
        for drug_x, protein_x, y in loader:
            drug_x = drug_x.to(device)
            protein_x = protein_x.to(device)
            pred = model(drug_x, protein_x).detach().cpu().numpy()
            preds.append(pred)
            ys.append(y.numpy())

    preds = np.concatenate(preds)
    ys = np.concatenate(ys)
    return ys, preds

In [13]:
def train_one_split(
    split_name,
    split,
    model_class,
    xy_builder=None,
    model_kwargs=None,
    epochs=50,
    batch_size=256,
    lr=1e-4,
    weight_decay=1e-4,
    patience=10,
):
    if model_kwargs is None:
        model_kwargs = {}

    print(f"\n===== Training on split: {split_name} =====")

    if xy_builder is None:
        def xy_builder(dataframe):
            return build_xy(dataframe, drug_features, protein_features)

    drug_x_train, protein_x_train, y_train_raw = xy_builder(split["train"])
    drug_x_valid, protein_x_valid, y_valid_raw = xy_builder(split["valid"])
    drug_x_test, protein_x_test, y_test_raw = xy_builder(split["test"])

    y_mean = y_train_raw.mean()
    y_std = y_train_raw.std() + 1e-8
    y_train = (y_train_raw - y_mean) / y_std
    y_valid = (y_valid_raw - y_mean) / y_std
    y_test = (y_test_raw - y_mean) / y_std

    train_loader = DataLoader(
        DTADataset(drug_x_train, protein_x_train, y_train),
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
    )
    valid_loader = DataLoader(
        DTADataset(drug_x_valid, protein_x_valid, y_valid),
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )
    test_loader = DataLoader(
        DTADataset(drug_x_test, protein_x_test, y_test),
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )

    drug_dim = drug_x_train.shape[1]
    protein_dim = protein_x_train.shape[1]
    model = model_class(drug_dim=drug_dim, protein_dim=protein_dim, **model_kwargs).to(device)

    criterion = nn.MSELoss()
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
    )

    best_valid_loss = float("inf")
    best_state = None
    wait = 0
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        train_losses = []

        for drug_x, protein_x, y in train_loader:
            drug_x = drug_x.to(device)
            protein_x = protein_x.to(device)
            y = y.to(device)
            optimizer.zero_grad()
            pred = model(drug_x, protein_x)
            loss = criterion(pred, y)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        valid_losses = []

        with torch.no_grad():
            for drug_x, protein_x, y in valid_loader:
                drug_x = drug_x.to(device)
                protein_x = protein_x.to(device)
                y = y.to(device)
                pred = model(drug_x, protein_x)
                loss = criterion(pred, y)
                valid_losses.append(loss.item())

        train_loss = float(np.mean(train_losses))
        valid_loss = float(np.mean(valid_losses))
        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "valid_loss": valid_loss,
        })

        if valid_loss < best_valid_loss:
            best_valid_loss = valid_loss
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
        if epoch == 1 or epoch % 5 == 0:
            print(f"Epoch {epoch:03d} | train loss {train_loss:.4f} | valid loss {valid_loss:.4f}")
        if wait >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

    model.load_state_dict(best_state)

    _, valid_pred_std = predict(model, valid_loader, device)
    _, test_pred_std = predict(model, test_loader, device)
    valid_pred = valid_pred_std * y_std + y_mean
    test_pred = test_pred_std * y_std + y_mean
    valid_metrics = compute_metrics(y_valid_raw, valid_pred)
    test_metrics = compute_metrics(y_test_raw, test_pred)
    print("Valid metrics:", valid_metrics)
    print("Test metrics:", test_metrics)

    result = {
        "Split": split_name,
        **test_metrics,
        "BestValidLoss_standardized_MSE": best_valid_loss,
        "EpochsTrained": len(history),
    }

    history_df = pd.DataFrame(history)
    return result, model, history_df

In [14]:
MY_MODEL_CONFIG = {
    "epochs": 50,
    "batch_size": 256,
    "lr": 1e-4,
    "weight_decay": 1e-4,
    "patience": 10,
    "model_kwargs": {
        "hidden_dim": 512,
        "dropout": 0.2,
    },
}

RUN_MODEL = True
results = []
histories = {}
if RUN_MODEL:
    for split_name, split in splits.items():
        result, model, history_df = train_one_split(
            split_name=split_name,
            split=split,
            model_class=CrossAttention,
            **MY_MODEL_CONFIG,
        )

        result["Model"] = "DTAModel"
        results.append(result)
        histories[split_name] = history_df

    results_df = pd.DataFrame(results)
    results_df = results_df[
        ["Model", "Split", "RMSE", "MAE", "Pearson", "Spearman", "BestValidLoss_standardized_MSE", "EpochsTrained"]
    ]

    display(results_df)
    results_path = f"{DATASET_NAME.lower()}_DTA_model_results.csv"
    results_df.to_csv(results_path, index=False)
    print(f"Saved {results_path}")

else:
    print("RUN_MODEL is False. Edit the cells, then set RUN_MODEL = True.")


===== Training on split: cold_drug =====
Epoch 001 | train loss 0.7013 | valid loss 0.6884
Epoch 005 | train loss 0.3778 | valid loss 0.6034
Epoch 010 | train loss 0.2821 | valid loss 0.5574
Epoch 015 | train loss 0.2246 | valid loss 0.5364
Epoch 020 | train loss 0.1842 | valid loss 0.5391
Epoch 025 | train loss 0.1580 | valid loss 0.5326
Epoch 030 | train loss 0.1399 | valid loss 0.5278
Epoch 035 | train loss 0.1242 | valid loss 0.5298
Early stopping at epoch 37
Valid metrics: {'RMSE': 0.5987721681594849, 'MAE': 0.3840699791908264, 'Pearson': 0.6881055235862732, 'Spearman': 0.6781732045010564}
Test metrics: {'RMSE': 0.6250854730606079, 'MAE': 0.38599783182144165, 'Pearson': 0.6877577900886536, 'Spearman': 0.6771496835375753}


,Model,Split,RMSE,MAE,Pearson,Spearman,BestValidLoss_standardized_MSE,EpochsTrained
0,DTAModel,cold_drug,0.625085,0.385998,0.687758,0.67715,0.520568,37


Saved kiba_DTA_model_results.csv


In [61]:
SKLEARN_AVAILABLE = True
TOX21_CSV_PATH = Path("/content/tox21.csv") # Set this path manually.
NOVOMOLGEN_MODEL_ID = "chandar-lab/NovoMolGen_32M_SMILES_AtomWise"
NOVOMOLGEN_REVISION = "hf-checkpoint"

# Reward weights. Tune these after checking the scale of each term printed during RL.
LAMBDA_LRRK2 = 0.4 # lambda_1
LAMBDA_SIMILAR_KINASE = 0.4 # lambda_2
LAMBDA_TOX21 = 0.5 # lambda_3
LAMBDA_QED = 0.4 # lambda_4
LAMBDA_SA = 0.1 # lambda_5
INVALID_SMILES_REWARD = -10.0

# If None, use all 228 off-target kinases for the reward.
OFFTARGET_BINDING_SAMPLE_N = None

RL_OUTPUT_DIR = Path("novomolgen_lrrk2_rl")
RL_OUTPUT_DIR.mkdir(exist_ok=True)
RL_STEPS = 400
RL_BATCH_SIZE = 32
RL_LR = 1e-5
RL_MAX_NEW_TOKENS = 96
RL_TEMPERATURE = 1.0
RL_TOP_P = 0.95
RL_ADVANTAGE_EPS = 1e-8
RL_KL_ANCHOR_BETA = 0.001 # Keep the agent close to the pre-trained prior; increase if validity collapses.
RL_SAVE_EVERY = 25
RL_LOG_CSV = RL_OUTPUT_DIR / "rl_training_log.csv"
RL_BEST_CHECKPOINT_METRIC = "reward_mean"
RL_BEST_CHECKPOINT_MODE = "max"
RL_BEST_CHECKPOINT_DIR = RL_OUTPUT_DIR / "best_checkpoint"
RL_BEST_CHECKPOINT_SUMMARY_CSV = RL_BEST_CHECKPOINT_DIR / "best_checkpoint_summary.csv"

# Generation
FINAL_SAMPLE_N = 32
FINAL_BATCH_SIZE = 32
PRE_RL_SAMPLE_N = FINAL_SAMPLE_N
PRE_RL_BATCH_SIZE = FINAL_BATCH_SIZE
PRE_RL_SAMPLE_CSV = RL_OUTPUT_DIR / "pre_rl_novomolgen_samples_scored.csv"
PRE_RL_RANKED_CSV = RL_OUTPUT_DIR / "pre_rl_novomolgen_ranked_candidates.csv"
POST_RL_SAMPLE_CSV = RL_OUTPUT_DIR / "post_rl_novomolgen_samples_scored.csv"
FINAL_CANDIDATE_CSV = RL_OUTPUT_DIR / "lrrk2_offtarget_avoiding_candidates.csv"
RL_COMPARISON_CSV = RL_OUTPUT_DIR / "pre_post_rl_comparison_summary.csv"
RL_COMPARISON_DELTA_CSV = RL_OUTPUT_DIR / "pre_post_rl_comparison_delta.csv"
set_seed(SEED)

print("TOX21_CSV_PATH:", TOX21_CSV_PATH)
print("NOVOMOLGEN:", NOVOMOLGEN_MODEL_ID, "revision=", NOVOMOLGEN_REVISION)
print("RL_OUTPUT_DIR:", RL_OUTPUT_DIR.resolve())
print("PRE_RL_SAMPLE_N / FINAL_SAMPLE_N:", PRE_RL_SAMPLE_N, FINAL_SAMPLE_N)

TOX21_CSV_PATH: /content/tox21.csv
NOVOMOLGEN: chandar-lab/NovoMolGen_32M_SMILES_AtomWise revision= hf-checkpoint
RL_OUTPUT_DIR: /content/novomolgen_lrrk2_rl
PRE_RL_SAMPLE_N / FINAL_SAMPLE_N: 32 32


In [62]:
DTA_MODEL = model.to(device)
DTA_MODEL.eval()
DTA_SPLIT_NAME = "cold_drug"
DTA_Y_MEAN = float(splits[DTA_SPLIT_NAME]["train"]["Y"].values.astype(np.float32).mean())
DTA_Y_STD = float(splits[DTA_SPLIT_NAME]["train"]["Y"].values.astype(np.float32).std() + 1e-8)
DTA_CHECKPOINT_PATH = RL_OUTPUT_DIR / "kiba_crossattention_dta_predictor.pt"
torch.save({
    "state_dict": DTA_MODEL.state_dict(),
    "model_class": "CrossAttention",
    "drug_dim": DRUG_DIM,
    "protein_dim": PROTEIN_DIM,
    "drug_fp_bits": DRUG_FP_BITS,
    "drug_fp_radius": DRUG_FP_RADIUS,
    "protein_max_len": PROTEIN_MAX_LEN,
    "esm_repr_layer": ESM_REPR_LAYER,
    "y_mean": DTA_Y_MEAN,
    "y_std": DTA_Y_STD,
    "split_name": DTA_SPLIT_NAME,
    "model_kwargs": MY_MODEL_CONFIG.get("model_kwargs", {}),
}, DTA_CHECKPOINT_PATH)
print("Saved DTA predictor checkpoint:", DTA_CHECKPOINT_PATH)
print("DTA raw Y mean/std:", DTA_Y_MEAN, DTA_Y_STD)

LRRK2_TARGET_SEQUENCE = None
LRRK2_QUERY_REGEX = r"(Q5S007)"

def infer_lrrk2_target_sequence(kiba_df):
    search_col = "Target_ID"
    hits = []
    mask = kiba_df[search_col].astype(str).str.contains(LRRK2_QUERY_REGEX, case=False, regex=True, na=False)
    if mask.any():
        for seq in kiba_df.loc[mask, "Target"].dropna().unique().tolist():
            hits.append((search_col, seq))
    unique = []
    seen = set()
    for col, seq in hits:
        if seq not in seen:
            unique.append((col, seq))
            seen.add(seq)
    if len(unique) == 0:
        preview_cols = [c for c in search_cols if c in kiba_df.columns]
        preview = kiba_df[preview_cols].drop_duplicates().head(20) if len(preview_cols) else pd.DataFrame()
        print("Could not automatically find LRRK2 in KIBA metadata columns.")
        if len(preview):
            display(preview)
        raise ValueError(
            "Set LRRK2_TARGET_SEQUENCE manually to the KIBA Target sequence for LRRK2, then rerun this cell."
        )
    if len(unique) > 1:
        print("Multiple LRRK2-like target sequences were found. Using the first; inspect if needed.")
        for i, (col, seq) in enumerate(unique[:5]):
            print(i, "matched column:", col, "sequence length:", len(seq))
    return unique[0][1]

if LRRK2_TARGET_SEQUENCE is None:
    LRRK2_TARGET_SEQUENCE = infer_lrrk2_target_sequence(kiba_data)

ALL_KIBA_TARGET_SEQS = sorted(kiba_data["Target"].dropna().unique().tolist())
SIMILAR_KINASE_TARGET_SEQS = [seq for seq in ALL_KIBA_TARGET_SEQS if seq != LRRK2_TARGET_SEQUENCE]
print("On-target LRRK2 sequence length:", len(LRRK2_TARGET_SEQUENCE))
print("KIBA unique targets:", len(ALL_KIBA_TARGET_SEQS))
print("Similar kinase off-target count:", len(SIMILAR_KINASE_TARGET_SEQS))
assert len(SIMILAR_KINASE_TARGET_SEQS) == len(ALL_KIBA_TARGET_SEQS) - 1

Saved DTA predictor checkpoint: novomolgen_lrrk2_rl/kiba_crossattention_dta_predictor.pt
DTA raw Y mean/std: 11.72458267211914 0.8305219511849976
On-target LRRK2 sequence length: 2527
KIBA unique targets: 229
Similar kinase off-target count: 228


In [63]:
def canonicalize_smiles(smiles):
    if smiles is None or (isinstance(smiles, float) and np.isnan(smiles)):
        return None
    s = str(smiles).strip().replace(" ", "")
    if len(s) == 0:
        return None
    mol = Chem.MolFromSmiles(s)
    if mol is None:
        return None
    try:
        Chem.SanitizeMol(mol)
    except Exception:
        return None
    return Chem.MolToSmiles(mol, canonical=True)

def detect_smiles_column(df):
    return "smiles"

def detect_tox21_label_columns(df, smiles_col):
    exclude = {smiles_col, "mol_id"}
    numeric_cols = []
    for c in df.columns:
        if c in exclude:
            continue
        vals = pd.to_numeric(df[c], errors="coerce")
        non_na = vals.dropna()
        if len(non_na) == 0:
            continue
        uniq = set(non_na.unique().tolist())
        if uniq.issubset({0, 1, 0.0, 1.0}):
            numeric_cols.append(c)
    if len(numeric_cols) != 12:
        print("Detected label columns:", numeric_cols)
        raise ValueError(
            f"Expected 12 Tox21 binary label columns, found {len(numeric_cols)}. "
            "Set TOX21_TASKS manually to the 12 columns."
        )
    return numeric_cols

class Tox21Dataset(Dataset):
    def __init__(self, x, y):
        self.x = torch.tensor(x, dtype=torch.float32)
        y = y.astype(np.float32)
        self.mask = torch.tensor(~np.isnan(y), dtype=torch.float32)
        self.y = torch.tensor(np.nan_to_num(y, nan=0.0), dtype=torch.float32)
    def __len__(self):
        return self.x.shape[0]
    def __getitem__(self, idx):
        return self.x[idx], self.y[idx], self.mask[idx]

class Tox21MLP(nn.Module):
    def __init__(self, input_dim=1024, num_tasks=12, hidden_dim=512, dropout=0.25):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_tasks),
        )
    def forward(self, x):
        return self.net(x)

def masked_bce_with_logits(logits, targets, mask, pos_weight=None):
    loss = F.binary_cross_entropy_with_logits(
        logits,
        targets,
        pos_weight=pos_weight,
        reduction="none",
    )
    loss = loss * mask
    denom = mask.sum().clamp_min(1.0)
    return loss.sum() / denom

def compute_tox21_pos_weight(y_train):
    pos = np.nansum(y_train == 1, axis=0).astype(np.float32)
    neg = np.nansum(y_train == 0, axis=0).astype(np.float32)
    pos_weight = neg / np.maximum(pos, 1.0)
    pos_weight = np.clip(pos_weight, 1.0, 50.0)
    return torch.tensor(pos_weight, dtype=torch.float32, device=device)

def evaluate_tox21(model, loader, pos_weight=None):
    model.eval()
    losses, logits_all, y_all, mask_all = [], [], [], []
    with torch.no_grad():
        for x, y, mask in loader:
            x = x.to(device)
            y = y.to(device)
            mask = mask.to(device)
            logits = model(x)
            loss = masked_bce_with_logits(logits, y, mask, pos_weight=pos_weight)
            losses.append(float(loss.item()))
            logits_all.append(logits.detach().cpu().numpy())
            y_all.append(y.detach().cpu().numpy())
            mask_all.append(mask.detach().cpu().numpy())
    logits_all = np.concatenate(logits_all, axis=0)
    y_all = np.concatenate(y_all, axis=0)
    mask_all = np.concatenate(mask_all, axis=0).astype(bool)
    probs = 1.0 / (1.0 + np.exp(-logits_all))
    metrics = {"loss": float(np.mean(losses))}
    if SKLEARN_AVAILABLE:
        aucs, aps = [], []
        for j in range(y_all.shape[1]):
            m = mask_all[:, j]
            if m.sum() < 2 or len(np.unique(y_all[m, j])) < 2:
                continue
            aucs.append(roc_auc_score(y_all[m, j], probs[m, j]))
            aps.append(average_precision_score(y_all[m, j], probs[m, j]))
        metrics["mean_auroc"] = float(np.mean(aucs)) if len(aucs) else np.nan
        metrics["mean_auprc"] = float(np.mean(aps)) if len(aps) else np.nan
    return metrics

def load_and_featurize_tox21(csv_path):
    tox_df = pd.read_csv(csv_path)
    smiles_col = detect_smiles_column(tox_df)
    tox_tasks = detect_tox21_label_columns(tox_df, smiles_col)
    print("Tox21 SMILES column:", smiles_col)
    print("Tox21 tasks:", tox_tasks)
    tox_df = tox_df.copy()
    tox_df["canonical_smiles"] = tox_df[smiles_col].apply(canonicalize_smiles)
    before = len(tox_df)
    tox_df = tox_df.dropna(subset=["canonical_smiles"]).reset_index(drop=True)
    print(f"Valid Tox21 molecules: {len(tox_df):,}/{before:,}")
    x = np.stack([
        smiles_to_morgan_count_vec(s, radius=DRUG_FP_RADIUS, n_bits=DRUG_FP_BITS)
        for s in tqdm(tox_df["canonical_smiles"].tolist(), desc="Tox21 fingerprints")
    ]).astype(np.float32)
    y = tox_df[tox_tasks].apply(pd.to_numeric, errors="coerce").values.astype(np.float32)
    return tox_df, smiles_col, tox_tasks, x, y

In [64]:
TOX21_MODEL_PATH = RL_OUTPUT_DIR / "tox21_multitask_predictor.pt"
TRAIN_TOX21 = True
if TRAIN_TOX21:
    if not TOX21_CSV_PATH.exists():
        raise FileNotFoundError(
            f"tox21.csv was not found at {TOX21_CSV_PATH}. Put tox21.csv next to the notebook or update TOX21_CSV_PATH."
        )

    tox_df, TOX_SMILES_COL, TOX21_TASKS, tox_x, tox_y = load_and_featurize_tox21(TOX21_CSV_PATH)
    rng = np.random.default_rng(SEED)
    indices = np.arange(len(tox_df))
    rng.shuffle(indices)
    n = len(indices)
    n_train = int(0.8 * n)
    n_valid = int(0.1 * n)
    train_idx = indices[:n_train]
    valid_idx = indices[n_train:n_train + n_valid]
    test_idx = indices[n_train + n_valid:]

    tox_train_loader = DataLoader(
        Tox21Dataset(tox_x[train_idx], tox_y[train_idx]),
        batch_size=256,
        shuffle=True,
        num_workers=0,
    )
    tox_valid_loader = DataLoader(
        Tox21Dataset(tox_x[valid_idx], tox_y[valid_idx]),
        batch_size=512,
        shuffle=False,
        num_workers=0,
    )
    tox_test_loader = DataLoader(
        Tox21Dataset(tox_x[test_idx], tox_y[test_idx]),
        batch_size=512,
        shuffle=False,
        num_workers=0,
    )

    tox_pos_weight = compute_tox21_pos_weight(tox_y[train_idx])
    tox_model = Tox21MLP(input_dim=DRUG_FP_BITS, num_tasks=len(TOX21_TASKS), hidden_dim=512, dropout=0.25).to(device)
    tox_optimizer = torch.optim.AdamW(tox_model.parameters(), lr=1e-3, weight_decay=1e-4)
    best_valid_loss = float("inf")
    best_state = None
    wait = 0
    TOX_EPOCHS = 100
    TOX_PATIENCE = 15
    tox_history = []

    for epoch in range(1, TOX_EPOCHS + 1):
        tox_model.train()
        train_losses = []
        for x, y, mask in tox_train_loader:
            x = x.to(device)
            y = y.to(device)
            mask = mask.to(device)
            tox_optimizer.zero_grad(set_to_none=True)
            logits = tox_model(x)
            loss = masked_bce_with_logits(logits, y, mask, pos_weight=tox_pos_weight)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(tox_model.parameters(), 5.0)
            tox_optimizer.step()
            train_losses.append(float(loss.item()))

        valid_metrics = evaluate_tox21(tox_model, tox_valid_loader, pos_weight=tox_pos_weight)
        row = {"epoch": epoch, "train_loss": float(np.mean(train_losses)), **{f"valid_{k}": v for k, v in valid_metrics.items()}}
        tox_history.append(row)

        if valid_metrics["loss"] < best_valid_loss:
            best_valid_loss = valid_metrics["loss"]
            best_state = copy.deepcopy(tox_model.state_dict())
            wait = 0
        else:
            wait += 1
        if epoch == 1 or epoch % 5 == 0:
            print(
                f"Epoch {epoch:03d} | train loss {row['train_loss']:.4f} | "
                f"valid loss {valid_metrics['loss']:.4f} | "
                f"valid AUROC {valid_metrics.get('mean_auroc', np.nan):.4f}"
            )
        if wait >= TOX_PATIENCE:
            print("Early stopping Tox21 at epoch", epoch)
            break

    tox_model.load_state_dict(best_state)
    tox_valid_metrics = evaluate_tox21(tox_model, tox_valid_loader, pos_weight=tox_pos_weight)
    tox_test_metrics = evaluate_tox21(tox_model, tox_test_loader, pos_weight=tox_pos_weight)
    print("Tox21 valid metrics:", tox_valid_metrics)
    print("Tox21 test metrics:", tox_test_metrics)
    tox_history_df = pd.DataFrame(tox_history)
    display(tox_history_df)
    tox_history_df.to_csv(RL_OUTPUT_DIR / "tox21_training_history.csv", index=False)
    torch.save({
        "state_dict": tox_model.state_dict(),
        "tox_tasks": TOX21_TASKS,
        "drug_fp_bits": DRUG_FP_BITS,
        "drug_fp_radius": DRUG_FP_RADIUS,
        "model_kwargs": {"input_dim": DRUG_FP_BITS, "num_tasks": len(TOX21_TASKS), "hidden_dim": 512, "dropout": 0.25},
        "valid_metrics": tox_valid_metrics,
        "test_metrics": tox_test_metrics,
    }, TOX21_MODEL_PATH)
    print("Saved Tox21 predictor:", TOX21_MODEL_PATH)

else:
    ckpt = torch.load(TOX21_MODEL_PATH, map_location=device)
    TOX21_TASKS = ckpt["tox_tasks"]
    tox_model = Tox21MLP(**ckpt["model_kwargs"]).to(device)
    tox_model.load_state_dict(ckpt["state_dict"])
    tox_model.eval()
    print("Loaded Tox21 predictor:", TOX21_MODEL_PATH)

Tox21 SMILES column: smiles
Tox21 tasks: ['NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 'NR-ER', 'NR-ER-LBD', 'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53']


[09:30:41] WARNING: not removing hydrogen atom without neighbors


Valid Tox21 molecules: 7,831/7,831


Tox21 fingerprints:   0%|          | 0/7831 [00:00<?, ?it/s]

[09:30:44] WARNING: not removing hydrogen atom without neighbors


Epoch 001 | train loss 1.1145 | valid loss 1.0813 | valid AUROC 0.7964
Epoch 005 | train loss 0.4442 | valid loss 1.1433 | valid AUROC 0.8174
Epoch 010 | train loss 0.2164 | valid loss 1.7287 | valid AUROC 0.8197
Epoch 015 | train loss 0.1393 | valid loss 1.8700 | valid AUROC 0.8227
Early stopping Tox21 at epoch 17
Tox21 valid metrics: {'loss': 1.0234323143959045, 'mean_auroc': 0.8255139760804443, 'mean_auprc': 0.4410464971586579}
Tox21 test metrics: {'loss': 0.952252596616745, 'mean_auroc': 0.8125934885240964, 'mean_auprc': 0.40435951906063966}


,epoch,train_loss,valid_loss,valid_mean_auroc,valid_mean_auprc
0,1,1.114526,1.081349,0.796404,0.404956
1,2,0.853501,1.023432,0.825514,0.441046
2,3,0.675839,1.058650,0.826219,0.446730
3,4,0.547131,1.076620,0.831843,0.459242
4,5,0.444211,1.143288,0.817406,0.451383
5,6,0.379121,1.240413,0.833124,0.463422
6,7,0.325109,1.373526,0.820770,0.467868
7,8,0.274960,1.486756,0.819667,0.460278
8,9,0.238175,1.456640,0.823379,0.454993
9,10,0.216401,1.728715,0.819715,0.451736


Saved Tox21 predictor: novomolgen_lrrk2_rl/tox21_multitask_predictor.pt


In [65]:
def mol_qed_sa(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return np.nan, np.nan
    try:
        qed = float(QED.qed(mol))
    except Exception:
        qed = np.nan
    try:
        sa = float(sascorer.calculateScore(mol))
    except Exception:
        sa = np.nan
    return qed, sa

@torch.no_grad()
def predict_tox21_prob_matrix(smiles_list, batch_size=1024):
    valid_smiles = [canonicalize_smiles(s) for s in smiles_list]
    out = np.full((len(smiles_list), len(TOX21_TASKS)), np.nan, dtype=np.float32)
    valid_indices = [i for i, s in enumerate(valid_smiles) if s is not None]
    if len(valid_indices) == 0:
        return out
    fps = np.stack([
        smiles_to_morgan_count_vec(valid_smiles[i], radius=DRUG_FP_RADIUS, n_bits=DRUG_FP_BITS)
        for i in valid_indices
    ]).astype(np.float32)
    tox_model.eval()
    probs = []
    for start in range(0, len(fps), batch_size):
        x = torch.tensor(fps[start:start + batch_size], dtype=torch.float32, device=device)
        logits = tox_model(x)
        probs.append(torch.sigmoid(logits).detach().cpu().numpy())
    probs = np.concatenate(probs, axis=0)
    out[valid_indices] = probs
    return out

@torch.no_grad()
def predict_binding_matrix(smiles_list, target_sequences, batch_size=4096, smiles_chunk_size=32):
    """Return raw-scale KIBA predictions with shape [num_smiles, num_targets]."""
    canonical = [canonicalize_smiles(s) for s in smiles_list]
    out = np.full((len(smiles_list), len(target_sequences)), np.nan, dtype=np.float32)
    valid_indices = [i for i, s in enumerate(canonical) if s is not None]
    if len(valid_indices) == 0 or len(target_sequences) == 0:
        return out
    target_arr = np.stack([protein_features[seq] for seq in target_sequences]).astype(np.float32)
    DTA_MODEL.eval()
    for idx_start in range(0, len(valid_indices), smiles_chunk_size):
        idx_chunk = valid_indices[idx_start:idx_start + smiles_chunk_size]
        drug_arr = np.stack([
            smiles_to_morgan_count_vec(canonical[i], radius=DRUG_FP_RADIUS, n_bits=DRUG_FP_BITS)
            for i in idx_chunk
        ]).astype(np.float32)
        n_s = len(idx_chunk)
        n_t = len(target_sequences)
        pair_drug = np.repeat(drug_arr, n_t, axis=0)
        pair_target = np.tile(target_arr, (n_s, 1))
        pred_chunks = []
        for start in range(0, len(pair_drug), batch_size):
            d = torch.tensor(pair_drug[start:start + batch_size], dtype=torch.float32, device=device)
            p = torch.tensor(pair_target[start:start + batch_size], dtype=torch.float32, device=device)
            pred_std = DTA_MODEL(d, p)
            pred_raw = pred_std.detach().cpu().numpy() * DTA_Y_STD + DTA_Y_MEAN
            pred_chunks.append(pred_raw.astype(np.float32))
        pred = np.concatenate(pred_chunks, axis=0).reshape(n_s, n_t)
        out[idx_chunk, :] = pred
    return out

def choose_offtarget_sequences(sample_n=None):
    if sample_n is None or sample_n >= len(SIMILAR_KINASE_TARGET_SEQS):
        return SIMILAR_KINASE_TARGET_SEQS
    return random.sample(SIMILAR_KINASE_TARGET_SEQS, sample_n)

def score_smiles_with_reward(smiles_list, offtarget_sample_n=OFFTARGET_BINDING_SAMPLE_N, return_per_task_tox=True):
    """
    Reward = lambda_1 * LRRK2_binding
             - lambda_2 * mean_binding_to_KIBA_kinases_except_LRRK2
             - lambda_3 * mean_Tox21_toxicity_probability (mean over 12 Tox21 endpoints)
             + lambda_4 * QED
             - lambda_5 * SA_score
    Invalid SMILES get INVALID_SMILES_REWARD.
    """
    canonical = [canonicalize_smiles(s) for s in smiles_list]
    valid = np.array([s is not None for s in canonical], dtype=bool)
    df = pd.DataFrame({
        "raw_smiles": smiles_list,
        "canonical_smiles": canonical,
        "valid": valid,
    })
    qed_vals, sa_vals = [], []
    for s in canonical:
        if s is None:
            qed_vals.append(np.nan)
            sa_vals.append(np.nan)
        else:
            q, sa = mol_qed_sa(s)
            qed_vals.append(q)
            sa_vals.append(sa)
    df["qed"] = qed_vals
    df["sa_score"] = sa_vals
    tox_probs = predict_tox21_prob_matrix(canonical)
    df["tox21_mean_prob"] = np.nanmean(tox_probs, axis=1)
    if return_per_task_tox:
        for j, task in enumerate(TOX21_TASKS):
            df[f"tox21_{task}"] = tox_probs[:, j]
    on_mat = predict_binding_matrix(canonical, [LRRK2_TARGET_SEQUENCE])
    df["lrrk2_binding"] = on_mat[:, 0]
    off_targets = choose_offtarget_sequences(offtarget_sample_n)
    off_mat = predict_binding_matrix(canonical, off_targets)
    df["similar_kinase_mean_binding"] = np.nanmean(off_mat, axis=1)
    df["similar_kinase_n_used"] = len(off_targets)
    reward = (
        LAMBDA_LRRK2 * df["lrrk2_binding"].values
        - LAMBDA_SIMILAR_KINASE * df["similar_kinase_mean_binding"].values
        - LAMBDA_TOX21 * df["tox21_mean_prob"].values
        + LAMBDA_QED * df["qed"].values
        - LAMBDA_SA * df["sa_score"].values
    ).astype(np.float32)
    reward[~valid] = INVALID_SMILES_REWARD
    reward = np.nan_to_num(reward, nan=INVALID_SMILES_REWARD, posinf=INVALID_SMILES_REWARD, neginf=INVALID_SMILES_REWARD)
    df["reward"] = reward
    return df

# Quick sanity check with known simple molecules before starting RL.
san_check_smiles = ["CCO", "c1ccccc1", "not_a_smiles"] # ethanol / benzene
sanity_df = score_smiles_with_reward(san_check_smiles, offtarget_sample_n=OFFTARGET_BINDING_SAMPLE_N)
display(sanity_df[["raw_smiles", "canonical_smiles", "valid", "lrrk2_binding", "similar_kinase_mean_binding", "tox21_mean_prob", "qed", "sa_score", "reward"]])

[09:30:49] SMILES Parse Error: syntax error while parsing: not_a_smiles
[09:30:49] SMILES Parse Error: Failed parsing SMILES 'not_a_smiles' for input: 'not_a_smiles'


,raw_smiles,canonical_smiles,valid,lrrk2_binding,similar_kinase_mean_binding,tox21_mean_prob,qed,sa_score,reward
0,CCO,CCO,True,11.701188,11.259211,0.143673,0.406808,1.980257,0.069652
1,c1ccccc1,c1ccccc1,True,11.324338,10.994205,0.215063,0.442628,1.000000,0.101573
2,not_a_smiles,None,False,NaN,NaN,NaN,NaN,NaN,-10.000000


In [66]:
# 32M NovoMolGen model is used. The 157M/300M variants can be substituted.
novo_tokenizer = AutoTokenizer.from_pretrained(
    NOVOMOLGEN_MODEL_ID,
    revision=NOVOMOLGEN_REVISION,
)
novo_model = AutoModelForCausalLM.from_pretrained(
    NOVOMOLGEN_MODEL_ID,
    revision=NOVOMOLGEN_REVISION,
).to(device)

if novo_tokenizer.pad_token_id is None:
    novo_tokenizer.pad_token = novo_tokenizer.eos_token if novo_tokenizer.eos_token is not None else novo_tokenizer.bos_token
novo_model.config.pad_token_id = novo_tokenizer.pad_token_id
if novo_model.config.eos_token_id is None and novo_tokenizer.eos_token_id is not None:
    novo_model.config.eos_token_id = novo_tokenizer.eos_token_id

novo_prior = AutoModelForCausalLM.from_pretrained(
    NOVOMOLGEN_MODEL_ID,
    revision=NOVOMOLGEN_REVISION,
).to(device)
novo_prior.eval()
for p in novo_prior.parameters():
    p.requires_grad_(False)

print("NovoMolGen tokenizer vocab size:", len(novo_tokenizer))
print("BOS/EOS/PAD:", novo_tokenizer.bos_token_id, novo_tokenizer.eos_token_id, novo_tokenizer.pad_token_id)

def decode_generated_smiles(sequences):
    decoded = novo_tokenizer.batch_decode(sequences, skip_special_tokens=True)
    smiles = [s.replace(" ", "").strip() for s in decoded]
    return smiles

@torch.no_grad()
def generate_novomolgen_samples(model, batch_size=16, max_new_tokens=96, temperature=1.0, top_p=0.95):
    model.eval()
    bos_id = novo_tokenizer.bos_token_id
    if bos_id is None:
        raise ValueError("Tokenizer has no bos_token_id; set a prompt/tokenization strategy manually.")
    input_ids = torch.full((batch_size, 1), bos_id, dtype=torch.long, device=device)
    sequences = model.generate(
        input_ids=input_ids,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        max_new_tokens=max_new_tokens,
        pad_token_id=novo_tokenizer.pad_token_id,
        eos_token_id=novo_tokenizer.eos_token_id,
    )
    return decode_generated_smiles(sequences), sequences

def sample_many(model, n, batch_size=32, max_new_tokens=96, temperature=1.0, top_p=0.95, desc="NovoMolGen sampling"):
    all_smiles = []
    pbar = tqdm(total=n, desc=desc)
    while len(all_smiles) < n:
        b = min(batch_size, n - len(all_smiles))
        smiles, _ = generate_novomolgen_samples(
            model,
            batch_size=b,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
        )
        all_smiles.extend(smiles)
        pbar.update(b)
    pbar.close()
    return all_smiles

GENERATION_DISPLAY_COLS = [
    "canonical_smiles", "reward", "lrrk2_binding", "similar_kinase_mean_binding",
    "tox21_mean_prob", "qed", "sa_score", "similar_kinase_n_used",
]

# Sample before RL.
pre_rl_smiles = sample_many(
    novo_model,
    n=PRE_RL_SAMPLE_N,
    batch_size=PRE_RL_BATCH_SIZE,
    max_new_tokens=RL_MAX_NEW_TOKENS,
    temperature=RL_TEMPERATURE,
    top_p=RL_TOP_P,
    desc="Pre-RL NovoMolGen sampling",
)
pre_rl_df = score_smiles_with_reward(
    pre_rl_smiles,
    offtarget_sample_n=OFFTARGET_BINDING_SAMPLE_N,
    return_per_task_tox=True,
)
pre_rl_ranked = (
    pre_rl_df[pre_rl_df["valid"]]
    .sort_values("reward", ascending=False)
    .drop_duplicates(subset=["canonical_smiles"])
    .reset_index(drop=True)
)
pre_rl_df.to_csv(PRE_RL_SAMPLE_CSV, index=False)
pre_rl_ranked.to_csv(PRE_RL_RANKED_CSV, index=False)
display(pre_rl_ranked[GENERATION_DISPLAY_COLS].head(50))
print("Saved pre-RL scored samples:", PRE_RL_SAMPLE_CSV)
print("Saved pre-RL ranked candidates:", PRE_RL_RANKED_CSV)
print("Pre-RL valid unique candidates:", len(pre_rl_ranked), "/", PRE_RL_SAMPLE_N)
print("Pre-RL validity:", float(pre_rl_df["valid"].mean()))

NovoMolGen tokenizer vocab size: 84
BOS/EOS/PAD: 2 3 1


Pre-RL NovoMolGen sampling:   0%|          | 0/32 [00:00<?, ?it/s]

,canonical_smiles,reward,lrrk2_binding,similar_kinase_mean_binding,tox21_mean_prob,qed,sa_score,similar_kinase_n_used
0,CN(C[C@@H]1CN(C(=O)c2cnc(C#N)s2)CCO1)C(=O)[C@H...,0.125440,12.222262,11.300246,0.225553,0.698426,4.099606,228
1,C=CCn1cc(C(=O)N2CCC[C@H](CN(C)C(=O)CC[C@H]3CCO...,0.086459,12.078810,11.389224,0.195376,0.640491,3.478839,228
2,Cc1ccc(C(=O)N2CC(C)(C)C[C@H]2CC(=O)N2CCC[C@@H]...,0.080803,11.608263,11.127945,0.199094,0.811564,3.364030,228
3,CCO[C@@H](C)c1nc(-c2cc(Br)c3[nH]cnc3c2)n(-c2cc...,0.062712,12.558976,11.383281,0.546252,0.533674,3.479099,228
4,Cc1cccc(C(=O)N2CC(F)(C(=O)N[C@@H](C)Cc3ccccn3)...,0.048459,11.408865,11.110441,0.213789,0.875747,3.143152,228
5,CC(C)C(=O)N[C@H]1C[C@H](C)CN(C(=O)c2ccc(O)c(F)...,0.029638,11.609077,11.402926,0.181692,0.886959,3.167598,228
6,CN=C(NCCCCCCC(C)C)NCc1cc(C)ccn1,0.001553,11.660226,11.272825,0.139540,0.415691,2.499137,228
7,Cc1cc(NC(=O)[C@@H](C)OC(=O)C[C@H](N2CCOCC2)C(F...,-0.004986,11.179377,10.845662,0.168792,0.748942,3.536528,228
8,C[C@H](OCC(F)(F)F)C(=O)NC[C@@H]1CN(C(=O)c2coc(...,-0.010753,11.544928,11.116231,0.232719,0.725326,3.560019,228
9,C[C@H](NC(=O)[C@H]1C[C@@H](F)CN(C(=O)c2cnccc2N...,-0.024029,11.713600,11.449499,0.207032,0.834189,3.598295,228


Saved pre-RL scored samples: novomolgen_lrrk2_rl/pre_rl_novomolgen_samples_scored.csv
Saved pre-RL ranked candidates: novomolgen_lrrk2_rl/pre_rl_novomolgen_ranked_candidates.csv
Pre-RL valid unique candidates: 32 / 32
Pre-RL validity: 1.0


In [67]:
def sequence_log_probs(model, sequences, prompt_len=1):
    """Teacher-forced log-probability of generated tokens under `model`."""
    sequences = sequences.to(device)
    attention_mask = (sequences != novo_tokenizer.pad_token_id).long()
    input_ids = sequences[:, :-1]
    labels = sequences[:, 1:]
    outputs = model(input_ids=input_ids, attention_mask=attention_mask[:, :-1])
    logits = outputs.logits.float()
    token_logp = F.log_softmax(logits, dim=-1).gather(-1, labels.unsqueeze(-1)).squeeze(-1)
    label_mask = attention_mask[:, 1:].float()
    if prompt_len > 1:
        label_mask[:, :prompt_len - 1] = 0.0
    seq_logp = (token_logp * label_mask).sum(dim=1)
    token_counts = label_mask.sum(dim=1).clamp_min(1.0)
    return seq_logp, token_counts

@torch.no_grad()
def prior_sequence_log_probs(sequences, prompt_len=1):
    novo_prior.eval()
    seq_logp, token_counts = sequence_log_probs(novo_prior, sequences, prompt_len=prompt_len)
    return seq_logp.detach(), token_counts.detach()

def _numeric_col(df, col):
    return pd.to_numeric(df[col], errors="coerce")

def summarize_reward_df(df):
    valid = df["valid"].fillna(False).astype(bool)
    reward = _numeric_col(df, "reward")
    lrrk2 = _numeric_col(df, "lrrk2_binding")
    off = _numeric_col(df, "similar_kinase_mean_binding")
    tox = _numeric_col(df, "tox21_mean_prob")
    qed = _numeric_col(df, "qed")
    sa = _numeric_col(df, "sa_score")
    return {
        "validity": float(valid.mean()),
        "valid_n": int(valid.sum()),
        "reward_mean": float(reward.mean()),
        "reward_max": float(reward.max()),
        "lrrk2_binding_mean": float(lrrk2.mean(skipna=True)),
        "lrrk2_binding_max": float(lrrk2.max(skipna=True)),
        "similar_kinase_binding_mean": float(off.mean(skipna=True)),
        "similar_kinase_binding_min": float(off.min(skipna=True)),
        "tox21_mean_prob": float(tox.mean(skipna=True)),
        "tox21_mean_prob_min": float(tox.min(skipna=True)),
        "qed_mean": float(qed.mean(skipna=True)),
        "qed_max": float(qed.max(skipna=True)),
        "sa_mean": float(sa.mean(skipna=True)),
        "sa_min": float(sa.min(skipna=True)),
    }

def summarize_generation_df(df, phase, sample_n=None):
    if sample_n is None:
        sample_n = len(df)
    valid = df["valid"].fillna(False).astype(bool)
    valid_unique_n = int(df.loc[valid, "canonical_smiles"].dropna().astype(str).nunique())
    s = summarize_reward_df(df)
    return {
        "phase": phase,
        "sample_n": int(sample_n),
        "valid_fraction": s["validity"],
        "valid_n": s["valid_n"],
        "valid_unique_n": valid_unique_n,
        "reward_mean_all_samples": s["reward_mean"],
        "reward_max": s["reward_max"],
        "lrrk2_binding_mean": s["lrrk2_binding_mean"],
        "lrrk2_binding_max": s["lrrk2_binding_max"],
        "similar_kinase_mean_binding_mean": s["similar_kinase_binding_mean"],
        "similar_kinase_mean_binding_min": s["similar_kinase_binding_min"],
        "tox21_mean_prob_mean": s["tox21_mean_prob"],
        "tox21_mean_prob_min": s["tox21_mean_prob_min"],
        "qed_mean": s["qed_mean"],
        "qed_max": s["qed_max"],
        "sa_score_mean": s["sa_mean"],
        "sa_score_min": s["sa_min"],
    }

def build_pre_post_comparison(pre_df, post_df):
    comparison_df = pd.DataFrame([
        summarize_generation_df(pre_df, "Before RL", sample_n=PRE_RL_SAMPLE_N),
        summarize_generation_df(post_df, "After RL", sample_n=FINAL_SAMPLE_N),
    ])
    numeric_cols = [c for c in comparison_df.columns if c != "phase"]
    delta = {"phase": "After - Before"}
    for c in numeric_cols:
        delta[c] = comparison_df.loc[1, c] - comparison_df.loc[0, c]
    delta_df = pd.DataFrame([delta])
    return comparison_df, delta_df

def is_better_checkpoint(value, best_value=None, mode="max"):
    if value is None or not np.isfinite(float(value)):
        return False
    if best_value is None:
        return True
    if mode == "max":
        return float(value) > float(best_value)
    if mode == "min":
        return float(value) < float(best_value)
    raise ValueError("RL_BEST_CHECKPOINT_MODE must be either 'max' or 'min'.")

def clone_model_state_to_cpu(model):
    return {name: tensor.detach().cpu().clone() for name, tensor in model.state_dict().items()}

def train_novomolgen_with_reward(
    model,
    steps=100,
    batch_size=16,
    lr=1e-6,
    max_new_tokens=96,
    temperature=1.0,
    top_p=0.95,
):
    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.0)
    logs = []
    baseline = None
    best_metric_value = None
    best_step = None
    best_summary = None
    best_state_dict = None
    for step in range(1, steps + 1):
        model.eval()
        smiles, sequences = generate_novomolgen_samples(
            model,
            batch_size=batch_size,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
        )
        reward_df = score_smiles_with_reward(smiles, offtarget_sample_n=OFFTARGET_BINDING_SAMPLE_N)
        rewards = torch.tensor(reward_df["reward"].values, dtype=torch.float32, device=device)
        batch_reward_mean = rewards.mean().detach()
        if baseline is None:
            baseline = batch_reward_mean
        else:
            baseline = 0.9 * baseline + 0.1 * batch_reward_mean
        advantages = rewards - baseline
        advantages = (advantages - advantages.mean()) / (advantages.std(unbiased=False) + RL_ADVANTAGE_EPS)
        model.train()
        seq_logp, token_counts = sequence_log_probs(model, sequences, prompt_len=1)
        with torch.no_grad():
            prior_logp, _ = prior_sequence_log_probs(sequences, prompt_len=1)
        policy_loss = -(advantages.detach() * seq_logp).mean()
        kl_anchor_loss = ((seq_logp - prior_logp) / token_counts).pow(2).mean()
        loss = policy_loss + RL_KL_ANCHOR_BETA * kl_anchor_loss
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        summary = summarize_reward_df(reward_df)
        summary.update({
            "step": step,
            "loss": float(loss.item()),
            "policy_loss": float(policy_loss.item()),
            "kl_anchor_loss": float(kl_anchor_loss.item()),
            "baseline": float(baseline.item()),
            "mean_seq_logp": float(seq_logp.mean().detach().cpu().item()),
            "best_smiles": reward_df.sort_values("reward", ascending=False).iloc[0]["canonical_smiles"],
        })
        current_metric = summary.get(RL_BEST_CHECKPOINT_METRIC)
        is_new_best = is_better_checkpoint(
            current_metric,
            best_metric_value,
            mode=RL_BEST_CHECKPOINT_MODE,
        )
        if is_new_best:
            best_metric_value = float(current_metric)
            best_step = step
            best_summary = dict(summary)
            best_state_dict = clone_model_state_to_cpu(model)
            RL_BEST_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
            model.save_pretrained(RL_BEST_CHECKPOINT_DIR)
            novo_tokenizer.save_pretrained(RL_BEST_CHECKPOINT_DIR)
            pd.DataFrame([best_summary]).to_csv(RL_BEST_CHECKPOINT_SUMMARY_CSV, index=False)
        summary["is_current_best"] = bool(is_new_best)
        summary["best_step_so_far"] = int(best_step) if best_step is not None else None
        summary["best_metric_so_far"] = float(best_metric_value) if best_metric_value is not None else np.nan
        logs.append(summary)
        if step == 1 or step % 5 == 0:
            print(
                f"Step {step:04d} | reward mean {summary['reward_mean']:.3f} | max {summary['reward_max']:.3f} | "
                f"valid {summary['validity']:.2f} | LRRK2 {summary['lrrk2_binding_mean']:.3f} | "
                f"off {summary['similar_kinase_binding_mean']:.3f} | tox {summary['tox21_mean_prob']:.3f} | "
                f"QED {summary['qed_mean']:.3f} | SA {summary['sa_mean']:.3f}"
            )
            print("  best:", summary["best_smiles"])
            print(
                f"  best checkpoint so far: step {best_step:04d} | "
                f"{RL_BEST_CHECKPOINT_METRIC}={best_metric_value:.4f}"
            )
        if step % RL_SAVE_EVERY == 0 or step == steps:
            ckpt_dir = RL_OUTPUT_DIR / f"checkpoint_step_{step:04d}"
            ckpt_dir.mkdir(exist_ok=True)
            model.save_pretrained(ckpt_dir)
            novo_tokenizer.save_pretrained(ckpt_dir)
            pd.DataFrame(logs).to_csv(RL_LOG_CSV, index=False)
            print("Saved checkpoint:", ckpt_dir)
            print("Current best checkpoint:", RL_BEST_CHECKPOINT_DIR)
    log_df = pd.DataFrame(logs)
    if best_step is not None:
        log_df["is_selected_best_checkpoint"] = log_df["step"].eq(best_step)
        model.load_state_dict(best_state_dict)
        model.to(device)
        model.eval()
        print(
            f"Restored best checkpoint from step {best_step:04d} "
            f"({RL_BEST_CHECKPOINT_METRIC}={best_metric_value:.4f}, mode={RL_BEST_CHECKPOINT_MODE})."
        )
        print("Best checkpoint saved at:", RL_BEST_CHECKPOINT_DIR)
    else:
        log_df["is_selected_best_checkpoint"] = False
        print("No valid best checkpoint was selected; returning the last model state.")
    log_df.to_csv(RL_LOG_CSV, index=False)
    return model, log_df

In [68]:
# This is the main RL training call. Increase RL_STEPS after checking reward scale and validity.
RUN_RL = True
if RUN_RL:
    novo_model, rl_log_df = train_novomolgen_with_reward(
        novo_model,
        steps=RL_STEPS,
        batch_size=RL_BATCH_SIZE,
        lr=RL_LR,
        max_new_tokens=RL_MAX_NEW_TOKENS,
        temperature=RL_TEMPERATURE,
        top_p=RL_TOP_P,
    )
    rl_log_df
else:
    print("RUN_RL=False. Set RUN_RL=True to train NovoMolGen with the reward.")

Step 0001 | reward mean -0.085 | max 0.251 | valid 1.00 | LRRK2 11.457 | off 11.176 | tox 0.217 | QED 0.659 | SA 3.523
  best: Cc1cc(C(=O)NCCN(C(=O)C2CCC2)C(C)C)ncn1
  best checkpoint so far: step 0001 | reward_mean=-0.0847
Step 0005 | reward mean -0.074 | max 0.263 | valid 1.00 | LRRK2 11.525 | off 11.182 | tox 0.249 | QED 0.692 | SA 3.636
  best: CN1CCC[C@](O)(CC(=O)NC[C@H](CO)CNC(=O)c2cnc3cccnn23)C1
  best checkpoint so far: step 0004 | reward_mean=-0.0701
Step 0010 | reward mean -0.139 | max 0.029 | valid 1.00 | LRRK2 11.390 | off 11.139 | tox 0.246 | QED 0.640 | SA 3.719
  best: O=C(Cc1ccc(Cl)cc1)N[C@@H]1CCN(C(=O)c2cnc3c(c2)COC3)C[C@@H]1F
  best checkpoint so far: step 0009 | reward_mean=-0.0442
Step 0015 | reward mean -0.119 | max 0.063 | valid 1.00 | LRRK2 11.500 | off 11.188 | tox 0.260 | QED 0.653 | SA 3.755
  best: CCCC1(O)CN(C(=O)[C@H]2CCN(C(=O)[C@@H]3CCOCC3(C)C)CCO2)C1
  best checkpoint so far: step 0009 | reward_mean=-0.0442
Step 0020 | reward mean -0.087 | max 0.100 | val

[09:35:57] SMILES Parse Error: extra close parentheses while parsing: CCOC(C)(C)C(=O)NCc1ncccc1C(=O)NC[C@@H](CC)n1cccn1)c1ccccc1F
[09:35:57] SMILES Parse Error: Failed parsing SMILES 'CCOC(C)(C)C(=O)NCc1ncccc1C(=O)NC[C@@H](CC)n1cccn1)c1ccccc1F' for input: 'CCOC(C)(C)C(=O)NCc1ncccc1C(=O)NC[C@@H](CC)n1cccn1)c1ccccc1F'


Step 0235 | reward mean -0.050 | max 0.165 | valid 1.00 | LRRK2 11.544 | off 11.240 | tox 0.253 | QED 0.701 | SA 3.256
  best: CCNC(=O)C1CCN(c2ccncc2S(=O)(=O)NCCCCCO)CC1
  best checkpoint so far: step 0135 | reward_mean=0.0051
Step 0240 | reward mean -0.043 | max 0.244 | valid 1.00 | LRRK2 11.459 | off 11.154 | tox 0.226 | QED 0.713 | SA 3.372
  best: Cc1nnc(C(=O)N2CCN(C(=O)c3ccccn3)CC2)s1
  best checkpoint so far: step 0135 | reward_mean=0.0051
Step 0245 | reward mean 0.005 | max 0.354 | valid 1.00 | LRRK2 11.594 | off 11.216 | tox 0.257 | QED 0.769 | SA 3.253
  best: CC1=NO[C@](C)(C(=O)N2CCC(NC(=O)c3nccc4ccncc34)CC2)C1
  best checkpoint so far: step 0135 | reward_mean=0.0051
Step 0250 | reward mean -0.048 | max 0.096 | valid 1.00 | LRRK2 11.467 | off 11.190 | tox 0.226 | QED 0.731 | SA 3.381
  best: CC1=C(C(=O)NCCN(C(=O)c2cnn3cccc(Cl)c23)C(C)C)CCC1
  best checkpoint so far: step 0135 | reward_mean=0.0051
Saved checkpoint: novomolgen_lrrk2_rl/checkpoint_step_0250
Current best checkpoi

[09:39:11] SMILES Parse Error: extra close parentheses while parsing: Cn1cncc1C(=O)N[C@H]1C[C@H]2CC[C@@H]1N2C(=O)c1cc(CO)ccc1Cl)C1
[09:39:11] SMILES Parse Error: Failed parsing SMILES 'Cn1cncc1C(=O)N[C@H]1C[C@H]2CC[C@@H]1N2C(=O)c1cc(CO)ccc1Cl)C1' for input: 'Cn1cncc1C(=O)N[C@H]1C[C@H]2CC[C@@H]1N2C(=O)c1cc(CO)ccc1Cl)C1'


Step 0395 | reward mean 0.053 | max 0.348 | valid 1.00 | LRRK2 11.759 | off 11.326 | tox 0.215 | QED 0.784 | SA 3.259
  best: CCN1CCC(CNc2ccc(C)c(-c3nnnn3C3CC3)c2)CC1
  best checkpoint so far: step 0389 | reward_mean=0.0625
Step 0400 | reward mean 0.024 | max 0.361 | valid 1.00 | LRRK2 11.690 | off 11.310 | tox 0.232 | QED 0.776 | SA 3.223
  best: CN1CC[C@@H](NC(=O)c2ncoc2CNC(=O)c2n[nH]cc2Cl)C1(C)C
  best checkpoint so far: step 0389 | reward_mean=0.0625
Saved checkpoint: novomolgen_lrrk2_rl/checkpoint_step_0400
Current best checkpoint: novomolgen_lrrk2_rl/best_checkpoint
Restored best checkpoint from step 0389 (reward_mean=0.0625, mode=max).
Best checkpoint saved at: novomolgen_lrrk2_rl/best_checkpoint


In [69]:
final_smiles = sample_many(
    novo_model,
    n=FINAL_SAMPLE_N,
    batch_size=FINAL_BATCH_SIZE,
    max_new_tokens=RL_MAX_NEW_TOKENS,
    temperature=RL_TEMPERATURE,
    top_p=RL_TOP_P,
    desc="Post-RL final NovoMolGen sampling",
)
final_df = score_smiles_with_reward(
    final_smiles,
    offtarget_sample_n=OFFTARGET_BINDING_SAMPLE_N,
    return_per_task_tox=True,
)
final_ranked = (
    final_df[final_df["valid"]]
    .sort_values("reward", ascending=False)
    .drop_duplicates(subset=["canonical_smiles"])
    .reset_index(drop=True)
)
final_df.to_csv(POST_RL_SAMPLE_CSV, index=False)
final_ranked.to_csv(FINAL_CANDIDATE_CSV, index=False)
print("Saved post-RL scored samples:", POST_RL_SAMPLE_CSV)
print("Saved final ranked candidates:", FINAL_CANDIDATE_CSV)
print("Post-RL valid unique candidates:", len(final_ranked), "/", FINAL_SAMPLE_N)
display(final_ranked[GENERATION_DISPLAY_COLS].head(50))

comparison_df, comparison_delta_df = build_pre_post_comparison(pre_rl_df, final_df)
comparison_df.to_csv(RL_COMPARISON_CSV, index=False)
comparison_delta_df.to_csv(RL_COMPARISON_DELTA_CSV, index=False)
print("Saved pre/post-RL comparison summary:", RL_COMPARISON_CSV)
print("Saved pre/post-RL comparison delta:", RL_COMPARISON_DELTA_CSV)
print("\nBefore vs After RL summary")
display(comparison_df)
print("\nAfter - Before delta")
display(comparison_delta_df)

FINAL_MODEL_DIR = RL_OUTPUT_DIR / "final_model"
FINAL_MODEL_DIR.mkdir(exist_ok=True)
novo_model.save_pretrained(FINAL_MODEL_DIR)
novo_tokenizer.save_pretrained(FINAL_MODEL_DIR)
print("Saved final NovoMolGen model:", FINAL_MODEL_DIR)

Post-RL final NovoMolGen sampling:   0%|          | 0/32 [00:00<?, ?it/s]

Saved post-RL scored samples: novomolgen_lrrk2_rl/post_rl_novomolgen_samples_scored.csv
Saved final ranked candidates: novomolgen_lrrk2_rl/lrrk2_offtarget_avoiding_candidates.csv
Post-RL valid unique candidates: 32 / 32


,canonical_smiles,reward,lrrk2_binding,similar_kinase_mean_binding,tox21_mean_prob,qed,sa_score,similar_kinase_n_used
0,CN(CC1CCN(C(=O)c2cccnc2)CC1)C(=O)c1cccc(O)c1,0.235121,11.740379,11.310463,0.177024,0.916010,2.147371,228
1,Cc1nnccc1C(=O)N1CC[C@@H](NC(=O)c2csc(-c3cn[nH]...,0.204772,12.971046,11.994447,0.279676,0.700823,3.263601,228
2,CC(C)Oc1ncccc1C(=O)N1CCCN(C(=O)CCn2ccnn2)CC1,0.185321,11.783798,11.216630,0.169556,0.741166,2.532347,228
3,Cc1nc(C)c(C)c(NC2CCN(C(=O)CN3CCCNC3=O)CC2)n1,0.170371,11.780630,11.352777,0.144496,0.842528,2.655334,228
4,CC1(C)CC(CC(=O)NCc2ccc(O)cc2)CC(C)(C)O1,0.165957,12.094793,11.433891,0.407637,0.896258,2.530888,228
5,Cc1ncsc1CCC(=O)N[C@H]1C[C@H]1NC(=O)c1cccnc1,0.163746,12.133858,11.487045,0.220977,0.839869,3.204381,228
6,COc1ncccc1CN1CC[C@@H](NC(=O)c2cncnc2)C(C)(C)C1,0.148721,11.951740,11.463688,0.167698,0.882459,3.156343,228
7,CC(C)COc1ccccc1NC(=O)c1ccncc1N1CCCC1,0.088090,11.691941,11.322339,0.387940,0.865094,2.118184,228
8,C=CCn1ccnc1C(=O)NC[C@H]1C[C@H]2CC[C@H]1N2C(=O)...,0.079253,12.643583,11.630967,0.212548,0.611425,4.640898,228
9,CCN(C[C@@H](C)NC(=O)c1cccc(CC#N)c1)C(=O)CCN1CC...,0.076479,11.879591,11.347342,0.249826,0.694119,2.891550,228


Saved pre/post-RL comparison summary: novomolgen_lrrk2_rl/pre_post_rl_comparison_summary.csv
Saved pre/post-RL comparison delta: novomolgen_lrrk2_rl/pre_post_rl_comparison_delta.csv

Before vs After RL summary


,phase,sample_n,valid_fraction,valid_n,valid_unique_n,reward_mean_all_samples,reward_max,lrrk2_binding_mean,lrrk2_binding_max,similar_kinase_mean_binding_mean,similar_kinase_mean_binding_min,tox21_mean_prob_mean,tox21_mean_prob_min,qed_mean,qed_max,sa_score_mean,sa_score_min
0,Before RL,32,1.0,32,32,-0.057134,0.125440,11.509933,12.558976,11.181256,10.323503,0.238652,0.098653,0.711629,0.886959,3.539299,2.499137
1,After RL,32,1.0,32,32,0.034934,0.235121,11.659334,12.971046,11.265255,10.424850,0.237455,0.129566,0.769281,0.916010,3.116824,2.118184



After - Before delta


,phase,sample_n,valid_fraction,valid_n,valid_unique_n,reward_mean_all_samples,reward_max,lrrk2_binding_mean,lrrk2_binding_max,similar_kinase_mean_binding_mean,similar_kinase_mean_binding_min,tox21_mean_prob_mean,tox21_mean_prob_min,qed_mean,qed_max,sa_score_mean,sa_score_min
0,After - Before,0,0.0,0,0,0.092068,0.109682,0.149401,0.41207,0.083999,0.101346,-0.001198,0.030912,0.057652,0.029051,-0.422475,-0.380953


Saved final NovoMolGen model: novomolgen_lrrk2_rl/final_model
